# Main.py

Ini buat Install Library

In [2]:
!pip install -qU \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-chroma \
    langchain-huggingface \
    pypdf \
    opentelemetry-api==1.38.0 \
    opentelemetry-sdk==1.38.0 \
    opentelemetry-exporter-otlp-proto-common==1.38.0 \
    opentelemetry-proto==1.38.0 \
    sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00


Ini buat import database

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('Dokumen_Tes_RAG_Rexaldo.pdf')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

retriever = vectorstore.as_retriever()

print("Database Dora the explorer berhasil dibuat yey >.<")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Database Dora the explorer berhasil dibuat yey >.<


Yang ini,eksekusi buat RAG

In [10]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

#PAKE API KEYNYA MASING-MASING
os.environ["GROQ_API_KEY"] = "Rahasia"

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

system_prompt = (
    "Kamu adalah asisten AI yang sangat cerdas.\n"
    "Gunakan data berikut untuk menjawab pertanyaan karyawan.\n"
    "Jika jawabannya tidak ada di data ini, bilang saja 'Data tidak ditemukan'.\n\n"
    "Konteks:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

Yang ini buat simpen chat history

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and the latest user question, formulate a standalone question. Do NOT answer it, just reformulate it."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()

def get_question(x):
    if x.get("chat_history"):
        return contextualize_chain.invoke(x)
    else:
        return x["input"]

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the following context to answer the question concisely:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnablePassthrough.assign(
        context=(lambda x: get_question(x)) | retriever | format_docs
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("AKHIRNYA berhasil")

AKHIRNYA berhasil


Yang ini buat output pertanyaan sama hasil

In [9]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

print("--- SESI CHAT DIMULAI ---\n")

question1 = "Apa topik utama dari dokumen ini?"
print(f"Rado: {question1}")

response1 = rag_chain.invoke({
    "input": question1,
    "chat_history": chat_history
})
print(f"AI: {response1}\n")

chat_history.extend([
    HumanMessage(content=question1),
    AIMessage(content=response1)
])

question2 = "Coba jelasin lebih detail lagi tentang hal itu."
print(f"Rado: {question2}")

response2 = rag_chain.invoke({
    "input": question2,
    "chat_history": chat_history
})
print(f"AI: {response2}")

--- SESI CHAT DIMULAI ---

Rado: Apa topik utama dari dokumen ini?
AI: Topik utama dari dokumen ini adalah "Panduan Operasional Karyawan" di PT. Maju Mundur AI, yang mencakup berbagai kebijakan dan prosedur yang berlaku untuk karyawan perusahaan.

Rado: Coba jelasin lebih detail lagi tentang hal itu.
AI: Dokumen "Panduan Operasional Karyawan" di PT. Maju Mundur AI adalah sebuah pedoman yang menjelaskan berbagai kebijakan dan prosedur yang berlaku untuk karyawan perusahaan. Dokumen ini mencakup beberapa aspek penting, seperti:

1. **Jam Kerja dan Lokasi**: Menjelaskan tentang jam kerja, lokasi kerja, dan keterlambatan.
2. **Struktur Gaji dan Bonus**: Menjelaskan tentang cara pembayaran gaji, bonus, dan insentif lainnya.
3. **Fasilitas Kantor**: Menjelaskan tentang fasilitas yang disediakan oleh perusahaan, seperti kopi gratis, makan siang, dan perpustakaan digital.
4. **Aturan Cuti Tambahan**: Menjelaskan tentang cuti tahunan, cuti khusus, dan syarat-syarat untuk mendapatkannya.
5. **Pr